# Stable Long-Tail Regression on TransLuxPop / CivitasGrid-TLP：Baseline vs Contrastive Learning（可复现实验 Notebook）

> 说明：本 Notebook **保留原有的数据处理 / 切分 / 指标汇报结构与超参**，仅将原先的“专家混合”章节替换为 **Contrastive Learning（改进损失函数）** 的建模与实验。
> 目标：在同一套 split 与同一套 metrics 下，让各组 reported errors 可以直接横向比较。

本 Notebook 实现：
- **Baseline**：XGBoost Regressor + RandomForest Regressor（分别建模 dVIIRS / dWorldPop）
- **Contrastive Learning 实验组（独立模型，单独评估）**
  - **XGB-CL**：在 XGBoost 上使用对比式回归目标（contrastive loss，负样本采样 + 温度 T）
  - **RT-CL**：在 RandomForest（Random Trees）上使用由对比式目标推导的 *contrastive reweighting*（sklearn RF 不支持自定义 objective，因此用样本权重实现同等“削弱 head、强调 tail”的训练偏好）
- **多种数据切分**：Random split / Group split by grid_id / Cross-domain split by region_type 或 city_type（自动降级）
- **评估**：全体 + Tail 子集 + worst-case（|error| 的 95/99 分位）
- **关键图**：预测-真值散点（tail 高亮）、残差分布、（可选）对比式诊断图（p_i 或权重分布、tail vs non-tail 均值对比）

数据默认读取：`/mnt/data/grids_set_4_HQ.xlsx`（grid-year records）。

---

## 与论文 Methodology / Contrastive Learning 章节对齐
- **Tail 定义**：训练集上计算每个目标的 5% 与 95% 分位阈值（避免 leakage），val/test 复用阈值
- **Baseline Model**：XGBoost（主基线）+ RandomForest（非 boosting 对照）
- **Contrastive Learning（改进损失函数）**：
  - 对每个样本 i，将预测 \(\hat{y}_i\) 作为 anchor，\(y_i\) 作为 positive；同 batch 的其它 \(y_j\) 作为 negatives
  - 使用距离 \(d(\hat{y}, y)\)（本 Notebook 默认 Smooth-L1）与温度 \(T\) 计算：

\[ p_i =
 rac{\exp\left(-d(\hat{y}_i,y_i)/T
ight)}
 {\exp\left(-d(\hat{y}_i,y_i)/T
ight)+\sum_{j
eq i}\exp\left(-d(\hat{y}_i,y_j)/T
ight)}
\]

  - 损失：\(\mathcal{L}_{con}=rac{1}{|\mathcal{B}|}\sum_i -\log p_i\)

> 注意：为保证可跑性与复杂度，本 Notebook 在对比式目标中采用 **固定数量的负样本采样**（每个 i 采样 K 个 negatives），并固定随机种子保证可复现。



In [1]:

# 本 cell：导入依赖、固定随机种子、配置全局可复现性（对应论文实验设置的可复现要求）。
import os
import random
import warnings
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GroupShuffleSplit, LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

import matplotlib.pyplot as plt

import xgboost as xgb

warnings.filterwarnings('ignore')

SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(SEED)

print('Versions:')
import sklearn
print('  pandas:', pd.__version__)
print('  numpy:', np.__version__)
print('  sklearn:', sklearn.__version__)
print('  xgboost:', xgb.__version__)


Versions:
  pandas: 2.3.3
  numpy: 2.3.5
  sklearn: 1.8.0
  xgboost: 3.1.2


In [2]:

# 本 cell：读取并初步检查 CivitasGrid-TLP / TransLuxPop（HQ）数据（grid-year records）。
# 为什么：确保列名、样本量、关键字段（grid_id/year/targets/features）存在且无明显异常。

data_path = 'grids_set_4_HQ.xlsx'
assert os.path.exists(data_path), f'找不到数据文件：{data_path}（请确认已上传）'

df = pd.read_excel(data_path)
df.columns = df.columns.str.strip()  # 清理尾部空格

print('Shape:', df.shape)
print('Has grid_id/year:', {'grid_id','year'}.issubset(df.columns))
print('Targets exist:', {'dVIIRS','dWorldPop'}.issubset(df.columns))

display(df.head(3))
# 在 Cell 2: 读完 df、清理列名后加入

import numpy as np

use_cols = [c for c in df.columns if c.endswith('_use')] + ["VIIRS_last_year", "WorldPop_last_year"]
df[use_cols] = df[use_cols] + np.random.uniform(-0.01, 0.01, size=df[use_cols].shape)


print("Rounded _use cols:", use_cols)
display(df[use_cols].head(3))


Shape: (28330, 37)
Has grid_id/year: True
Targets exist: True


,grid_id,year,nation_code,lat_min,lon_min,lat_max,lon_max,cell_area,region_type,city_type,...,City,description,Intersec_use,mot_use,tru_use,pri_use,sec_use,ter_use,urb_use,len_unc
0,1000,2015.0,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,...,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,108.242695,2.599565,0.850466,0.047647,1.75934,0.545176,5.066507,0
1,1000,2016.0,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,...,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,108.242695,2.599565,0.850466,0.047647,1.75934,0.545176,5.066507,0
2,1000,2017.0,USA,37.556543,-77.420981,37.572814,-77.40471,2.596018,commercial,developing,...,Richmond,发展中城市：市中心商业核心，路网密集/夜光强。,108.242695,2.599565,0.850466,0.047647,1.75934,0.545176,5.066507,0


Rounded _use cols: ['Intersec_use', 'mot_use', 'tru_use', 'pri_use', 'sec_use', 'ter_use', 'urb_use', 'VIIRS_last_year', 'WorldPop_last_year']


,Intersec_use,mot_use,tru_use,pri_use,sec_use,ter_use,urb_use,VIIRS_last_year,WorldPop_last_year
0,108.240186,2.608580,0.855106,0.049620,1.752460,0.538296,5.057669,NaN,NaN
1,108.246856,2.589977,0.859864,0.054296,1.753587,0.538813,5.060175,35.999961,929.313850
2,108.241334,2.595390,0.852703,0.040437,1.755183,0.542503,5.065629,30.526942,926.738134


In [3]:

# 本 cell：构造 covid_intensity 特征（按你给定的年份映射），并定义特征/目标列。
# 为什么：该特征属于外生冲击强度，按固定规则生成可复现。

covid_map = {
    2015: 0.0,
    2016: 0.0,
    2017: 0.0,
    2018: 0.0,
    2019: 0.05,
    2020: 0.8,
    2021: 1.0,
    2022: 0.6,
    2023: 0.4,
    2024: 0.2,
}

# year 极少缺失：无法构造 covid_intensity → 丢弃（可复现）
df = df.dropna(subset=['year']).copy()
df['year'] = df['year'].astype(int)
df['covid_intensity'] = df['year'].map(covid_map).fillna(0.0)

feature_cols = [
    'mot_use', 'tru_use', 'pri_use', 'sec_use', 'ter_use', 'urb_use',
    'VIIRS_last_year', 'WorldPop_last_year', 'covid_intensity',
    'region_type', 'city_type'
]

target_cols = ['dVIIRS', 'dWorldPop']

missing_cols = [c for c in feature_cols + target_cols + ['grid_id','year'] if c not in df.columns]
assert len(missing_cols) == 0, f'缺少列：{missing_cols}'

print('Feature cols:', feature_cols)
print('Targets:', target_cols)
print('Unique region_type:', df['region_type'].nunique(), 'Unique city_type:', df['city_type'].nunique())


Feature cols: ['mot_use', 'tru_use', 'pri_use', 'sec_use', 'ter_use', 'urb_use', 'VIIRS_last_year', 'WorldPop_last_year', 'covid_intensity', 'region_type', 'city_type']
Targets: ['dVIIRS', 'dWorldPop']
Unique region_type: 5 Unique city_type: 5


In [4]:

# 本 cell：缺失值处理策略说明与缺失概况。
# 为什么：你要求“若存在缺失需明确说明并可复现”。
# - 数值特征：中位数填补（仅在训练集 fit，防泄漏）
# - 类别特征：缺失填 'Unknown'（仅在训练集 fit，防泄漏）
# - 目标 y：建模时必须非缺失（每个目标分别过滤）

numeric_cols = [
    'mot_use','tru_use','pri_use','sec_use','ter_use','urb_use',
    'VIIRS_last_year','WorldPop_last_year','covid_intensity'
]

categorical_cols = ['region_type','city_type']

missing_summary = df[feature_cols + target_cols].isna().mean().sort_values(ascending=False)
display(missing_summary.to_frame('missing_ratio').head(12))


,missing_ratio
VIIRS_last_year,0.100004
dWorldPop,0.100004
dVIIRS,0.100004
WorldPop_last_year,0.100004
mot_use,0.000000
ter_use,0.000000
sec_use,0.000000
pri_use,0.000000
tru_use,0.000000
covid_intensity,0.000000


In [5]:

# 本 cell：定义预处理器（数值缺失填补 + 类别 One-Hot 编码）。
# 为什么：region_type/city_type 是类别特征，需要编码；其余为数值特征。

def build_preprocessor(numeric_features: List[str], categorical_features: List[str]) -> ColumnTransformer:
    try:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

    numeric_tf = Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])
    categorical_tf = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('onehot', ohe)
    ])

    pre = ColumnTransformer(
        transformers=[
            ('num', numeric_tf, numeric_features),
            ('cat', categorical_tf, categorical_features)
        ],
        remainder='drop',
        verbose_feature_names_out=False
    )
    return pre

preprocessor = build_preprocessor(numeric_cols, categorical_cols)


In [6]:
# 本 cell：定义 Contrastive Learning 所需的目标函数（对比式回归损失）/负样本采样，
# 并保留 benchmark 的 tail 阈值计算与评估指标（全体+tail+worst-case）。
# 为什么：对比学习实验组需要“改进损失函数”，同时要求 reported errors 与 baseline 可直接比较。

import numpy as np
from typing import Tuple, Dict

# -------------------------
# Contrastive helpers
# -------------------------

# 固定负样本采样个数（每个样本 i 采样 K 个 negatives）。
# 说明：K 越大越接近 full-batch contrastive，但训练会更慢。
CL_N_NEG = 16

# Smooth-L1 的数值稳定项（避免 |x| 在 0 处不可导，且避免除 0）。
CL_EPS = 1e-6

# 温度使用“训练集标准差 * 系数”的自适应方案，避免两目标尺度差异导致 exp 下溢/上溢。
CL_TEMP_FACTOR = 1.0


def sample_negative_indices(n: int, n_neg: int, seed: int = 42) -> np.ndarray:
    # 为每个样本 i 采样 n_neg 个 negatives 的索引（保证不等于 i）。
    # 向量化实现：先在 0..n-2 采样，再对 >= i 的索引整体 +1。
    if n <= 1:
        return np.zeros((n, n_neg), dtype=int)
    rng = np.random.default_rng(seed)
    neg = rng.integers(0, n - 1, size=(n, n_neg))  # 0..n-2
    row = np.arange(n)[:, None]
    neg = neg + (neg >= row)
    return neg.astype(int)


def smooth_abs(u: np.ndarray, eps: float = CL_EPS) -> np.ndarray:
    # Smooth L1: sqrt(u^2 + eps)
    return np.sqrt(u * u + eps)


def contrastive_prob(y_pred: np.ndarray, y_pos: np.ndarray, y_neg: np.ndarray, T: float, eps: float = CL_EPS) -> np.ndarray:
    # 计算对比式概率 p_i。
    # y_pred: (n,), y_pos: (n,), y_neg: (n,K)
    T = float(max(T, 1e-12))
    d_pos = smooth_abs(y_pred - y_pos, eps=eps)
    pos = np.exp(-d_pos / T)
    d_neg = smooth_abs(y_pred[:, None] - y_neg, eps=eps)
    neg = np.exp(-d_neg / T)
    denom = pos + np.sum(neg, axis=1)
    return pos / np.maximum(denom, 1e-12)


def make_xgb_contrastive_obj(neg_labels: np.ndarray, T: float, eps: float = CL_EPS):
    # 返回可直接传给 XGBRegressor(objective=...) 的自定义 objective。
    # 注意：XGBoost 需要每个样本的 grad/hess（对 y_pred_i），而本对比式目标满足“每个 i 的损失只依赖 y_pred_i”。
    T = float(max(T, 1e-12))

    def obj(y_true: np.ndarray, y_pred: np.ndarray):
        y_true = np.asarray(y_true, dtype=float)
        y_pred = np.asarray(y_pred, dtype=float)

        # ---- positive ----
        u_pos = y_pred - y_true
        d_pos = smooth_abs(u_pos, eps=eps)
        pos = np.exp(-d_pos / T)

        # ---- negatives (labels only) ----
        y_neg = neg_labels  # shape (n, K)
        u_neg = y_pred[:, None] - y_neg
        d_neg = smooth_abs(u_neg, eps=eps)
        neg = np.exp(-d_neg / T)
        neg_sum = np.sum(neg, axis=1)

        S = np.maximum(pos + neg_sum, 1e-12)
        pos_safe = np.maximum(pos, 1e-12)

        # ---- first derivatives ----
        dd_pos = u_pos / d_pos
        a_pos = (-1.0 / T) * dd_pos
        dpos = pos * a_pos

        dd_neg = u_neg / d_neg
        a_neg = (-1.0 / T) * dd_neg
        dneg = neg * a_neg
        dneg_sum = np.sum(dneg, axis=1)

        dS = dpos + dneg_sum

        # loss = log(S) - log(pos)
        grad = (dS / S) - (dpos / pos_safe)

        # ---- second derivatives ----
        d2d_pos = eps / (d_pos ** 3)
        a_pos_p = (-1.0 / T) * d2d_pos
        d2pos = pos * (a_pos * a_pos + a_pos_p)

        d2d_neg = eps / (d_neg ** 3)
        a_neg_p = (-1.0 / T) * d2d_neg
        d2neg = neg * (a_neg * a_neg + a_neg_p)
        d2neg_sum = np.sum(d2neg, axis=1)

        d2S = d2pos + d2neg_sum

        hess = (d2S / S) - (dS * dS) / (S * S) - (d2pos / pos_safe) + (dpos * dpos) / (pos_safe * pos_safe)
        hess = np.maximum(hess, 1e-6)

        return grad, hess

    return obj


def compute_contrastive_weights_from_labels(y: np.ndarray, n_neg: int, T: float, seed: int = 42) -> np.ndarray:
    # 给不支持自定义 objective 的模型（如 sklearn RandomForest）构造“对比式重加权”。
    # 思路：令正样本距离为 0（pos=1），仅用 label 间的负样本相似度衡量 label 的“拥挤程度”。
    # w_i = 1 / (1 + Σ_j exp(-|y_i - y_j|/T))，head 附近样本多 → 权重更小；tail 更稀疏 → 权重更大。
    y = np.asarray(y, dtype=float)
    n = len(y)
    if n <= 1:
        return np.ones_like(y, dtype=float)

    neg_idx = sample_negative_indices(n, n_neg, seed=seed)
    y_neg = y[neg_idx]

    d = np.abs(y[:, None] - y_neg)
    neg_sim = np.exp(-d / max(float(T), 1e-12))
    neg_sum = np.sum(neg_sim, axis=1)

    w = 1.0 / (1.0 + neg_sum)
    # 归一化到均值=1，避免样本权重尺度导致模型行为变化过大
    w = w / np.mean(w)
    w = np.clip(w, 0.05, 20.0)
    return w


def compute_tail_thresholds(y_train: np.ndarray, q_low: float = 0.05, q_high: float = 0.95) -> Tuple[float, float]:
    q05 = np.quantile(y_train, q_low)
    q95 = np.quantile(y_train, q_high)
    return float(q05), float(q95)


def tail_mask(y: np.ndarray, q05: float, q95: float) -> np.ndarray:
    y = np.asarray(y)
    return (y <= q05) | (y >= q95)


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred)
    abs_err = np.abs(y_true - y_pred)
    return {
        'MAE': float(mae),
        'RMSE': float(rmse),
        'R2': float(r2),
        'AbsErr_P95': float(np.quantile(abs_err, 0.95)),
        'AbsErr_P99': float(np.quantile(abs_err, 0.99)),
    }


def evaluate_with_tail(y_true: np.ndarray, y_pred: np.ndarray, q05: float, q95: float) -> Dict[str, float]:
    out = {}
    out.update({f'All_{k}': v for k, v in regression_metrics(y_true, y_pred).items()})
    m = tail_mask(y_true, q05, q95)
    out['Tail_Rate'] = float(m.mean())
    if m.sum() > 0:
        out.update({f'Tail_{k}': v for k, v in regression_metrics(y_true[m], y_pred[m]).items()})
    else:
        out.update({f'Tail_{k}': np.nan for k in ['MAE','RMSE','R2','AbsErr_P95','AbsErr_P99']})
    return out



In [7]:
# 本 cell：集中定义所有模型超参数（便于后续调参），并固定 random_state 保证可复现。
# 注意：你要求“不改模型超参”，因此 baseline 与实验组（CL）使用同一套树模型超参；
# 差异仅来自“损失函数/训练目标”（XGB 自定义 objective；RT 使用对比式权重）。

from dataclasses import dataclass
from typing import Dict

@dataclass
class HParams:
    xgb_reg: Dict
    rf_reg: Dict
    early_stopping_rounds: int

hparams = HParams(
    xgb_reg=dict(
        n_estimators=1200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        min_child_weight=1.0,
        objective='reg:squarederror',
        tree_method='hist',
        random_state=SEED,
        n_jobs=-1,
    ),
    rf_reg=dict(
        n_estimators=150,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=SEED,
        n_jobs=-1,
    ),
    early_stopping_rounds=30,
)

print(hparams)



HParams(xgb_reg={'n_estimators': 1200, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'min_child_weight': 1.0, 'objective': 'reg:squarederror', 'tree_method': 'hist', 'random_state': 42, 'n_jobs': -1}, rf_reg={'n_estimators': 150, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'random_state': 42, 'n_jobs': -1}, early_stopping_rounds=30)


In [8]:

# 本 cell：实现 3 类数据切分（random / group by grid_id / cross-domain by region_type 或 city_type）。
# 为什么：严格满足你要求的多种 split，并保证同一 split 下 Baseline 与 Contrastive Learning 使用同一 train/val/test。

def make_random_split(df_in: pd.DataFrame, test_size=0.15, val_size=0.15, seed=SEED):
    idx = np.arange(len(df_in))
    train_idx, temp_idx = train_test_split(idx, test_size=test_size+val_size, random_state=seed, shuffle=True)
    rel_test = test_size / (test_size + val_size)
    val_idx, test_idx = train_test_split(temp_idx, test_size=rel_test, random_state=seed, shuffle=True)
    return train_idx, val_idx, test_idx

def make_group_split_by_grid(df_in: pd.DataFrame, group_col='grid_id', test_size=0.15, val_size=0.15, seed=SEED):
    idx = np.arange(len(df_in))
    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_size+val_size, random_state=seed)
    train_idx, temp_idx = next(gss1.split(idx, groups=df_in[group_col].values))

    rel_test = test_size / (test_size + val_size)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=rel_test, random_state=seed)
    val_sub, test_sub = next(gss2.split(temp_idx, groups=df_in.iloc[temp_idx][group_col].values))
    val_idx = temp_idx[val_sub]
    test_idx = temp_idx[test_sub]
    return train_idx, val_idx, test_idx

def choose_crossdomain_group_col(df_in: pd.DataFrame) -> str:
    if 'region_type' in df_in.columns and df_in['region_type'].nunique() >= 2:
        return 'region_type'
    if 'city_type' in df_in.columns and df_in['city_type'].nunique() >= 2:
        return 'city_type'
    raise ValueError('没有可用于 cross-domain 的 group 字段（region_type/city_type 均不可用）')

def iter_leave_one_group_out(df_in: pd.DataFrame, group_col: str):
    logo = LeaveOneGroupOut()
    groups = df_in[group_col].values
    idx = np.arange(len(df_in))
    for train_val_idx, test_idx in logo.split(idx, groups=groups):
        test_group = df_in.iloc[test_idx][group_col].iloc[0]
        yield train_val_idx, test_idx, test_group

def make_val_split_within_train(df_in: pd.DataFrame, train_val_idx: np.ndarray, val_size=0.15, seed=SEED):
    sub = df_in.iloc[train_val_idx].copy()
    sub_idx = np.arange(len(sub))
    if 'grid_id' in sub.columns and sub['grid_id'].nunique() >= 2:
        gss = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
        tr_sub, va_sub = next(gss.split(sub_idx, groups=sub['grid_id'].values))
    else:
        tr_sub, va_sub = train_test_split(sub_idx, test_size=val_size, random_state=seed, shuffle=True)
    train_idx = train_val_idx[tr_sub]
    val_idx = train_val_idx[va_sub]
    return train_idx, val_idx

print('Split helpers ready.')


Split helpers ready.


In [9]:
# 本 cell：实现 Baseline（XGB/RT）与 Contrastive Learning 实验组（XGB-CL / RT-CL）的训练与推理。
# 为什么：封装“预处理（仅在训练集 fit）→ 模型训练（可 early stopping）→ 测试预测”，保证公平对比。

from dataclasses import dataclass
from typing import Dict, Optional

@dataclass
class FitArtifacts:
    y_true: np.ndarray
    y_pred: np.ndarray
    tail_q05: float
    tail_q95: float
    diag_test: Optional[np.ndarray] = None   # 诊断量：CL 模型存 p_i / 或权重 proxy；baseline 为 None
    diag_name: Optional[str] = None         # 诊断量名字（用于画图标题）


def fit_transform_features(pre: ColumnTransformer, X_train: pd.DataFrame, X_val: pd.DataFrame, X_test: pd.DataFrame):
    pre_fitted = pre.fit(X_train)
    Xt = pre_fitted.transform(X_train)
    Xv = pre_fitted.transform(X_val)
    Xs = pre_fitted.transform(X_test)
    return pre_fitted, Xt, Xv, Xs

# ---- XGBoost early stopping 兼容（保留原逻辑） ----
import inspect

def _xgb_fit_compat(model, Xt, y, Xv, yv, early_stopping_rounds: int):
    fit_sig = inspect.signature(model.fit)
    fit_params = fit_sig.parameters

    kwargs = {}
    if "eval_set" in fit_params:
        kwargs["eval_set"] = [(Xv, yv)]
    if "verbose" in fit_params:
        kwargs["verbose"] = False

    # 1) callbacks（新版本）
    if "callbacks" in fit_params and early_stopping_rounds and early_stopping_rounds > 0:
        kwargs["callbacks"] = [xgb.callback.EarlyStopping(rounds=early_stopping_rounds, save_best=True)]

    # 2) early_stopping_rounds（老版本）
    elif "early_stopping_rounds" in fit_params and early_stopping_rounds and early_stopping_rounds > 0:
        kwargs["early_stopping_rounds"] = early_stopping_rounds

    model.fit(Xt, y, **kwargs)
    return model


def train_xgb_regressor(Xt, yt, Xv, yv, params: Dict, early_stopping_rounds: int):
    model = xgb.XGBRegressor(**params)
    return _xgb_fit_compat(model, Xt, yt, Xv, yv, early_stopping_rounds)


def train_rt_regressor(Xt, yt, params: Dict, sample_weight=None):
    model = RandomForestRegressor(**params)
    if sample_weight is None:
        model.fit(Xt, yt)
    else:
        model.fit(Xt, yt, sample_weight=sample_weight)
    return model


def _compute_T_from_train(y_train: np.ndarray) -> float:
    # 自适应温度：std(y_train) * factor
    s = float(np.std(y_train))
    return max(s * float(CL_TEMP_FACTOR), 1e-6)


def _compute_contrastive_diag_on_test(y_test: np.ndarray, y_pred: np.ndarray, T: float, seed: int = SEED):
    # 诊断用：在 test 集内部采样 negatives，计算 p_i 分布
    n = len(y_test)
    if n <= 1:
        return None
    neg_idx = sample_negative_indices(n, CL_N_NEG, seed=seed)
    y_neg = y_test[neg_idx]
    p = contrastive_prob(y_pred, y_test, y_neg, T=T, eps=CL_EPS)
    return p


def run_baseline_xgb(df_in: pd.DataFrame, train_idx, val_idx, test_idx, target: str) -> FitArtifacts:
    sub_train = df_in.iloc[train_idx].dropna(subset=[target])
    sub_val = df_in.iloc[val_idx].dropna(subset=[target])
    sub_test = df_in.iloc[test_idx].dropna(subset=[target])

    X_train = sub_train[feature_cols]
    y_train = sub_train[target].values.astype(float)
    X_val = sub_val[feature_cols]
    y_val = sub_val[target].values.astype(float)
    X_test = sub_test[feature_cols]
    y_test = sub_test[target].values.astype(float)

    q05, q95 = compute_tail_thresholds(y_train, 0.05, 0.95)

    pre = build_preprocessor(numeric_cols, categorical_cols)
    pre, Xt, Xv, Xs = fit_transform_features(pre, X_train, X_val, X_test)

    model = train_xgb_regressor(Xt, y_train, Xv, y_val, hparams.xgb_reg, hparams.early_stopping_rounds)
    y_pred = model.predict(Xs)
    return FitArtifacts(y_true=y_test, y_pred=y_pred, tail_q05=q05, tail_q95=q95)


def run_baseline_rt(df_in: pd.DataFrame, train_idx, val_idx, test_idx, target: str) -> FitArtifacts:
    sub_train = df_in.iloc[train_idx].dropna(subset=[target])
    sub_val = df_in.iloc[val_idx].dropna(subset=[target])
    sub_test = df_in.iloc[test_idx].dropna(subset=[target])

    X_train = sub_train[feature_cols]
    y_train = sub_train[target].values.astype(float)
    X_val = sub_val[feature_cols]
    y_val = sub_val[target].values.astype(float)
    X_test = sub_test[feature_cols]
    y_test = sub_test[target].values.astype(float)

    q05, q95 = compute_tail_thresholds(y_train, 0.05, 0.95)

    pre = build_preprocessor(numeric_cols, categorical_cols)
    pre, Xt, Xv, Xs = fit_transform_features(pre, X_train, X_val, X_test)

    model = train_rt_regressor(Xt, y_train, hparams.rf_reg)
    y_pred = model.predict(Xs)
    return FitArtifacts(y_true=y_test, y_pred=y_pred, tail_q05=q05, tail_q95=q95)


def run_contrastive_xgb(df_in: pd.DataFrame, train_idx, val_idx, test_idx, target: str) -> FitArtifacts:
    # XGB-CL：在保持树结构超参不变的前提下，替换 objective 为 contrastive loss
    sub_train = df_in.iloc[train_idx].dropna(subset=[target])
    sub_val = df_in.iloc[val_idx].dropna(subset=[target])
    sub_test = df_in.iloc[test_idx].dropna(subset=[target])

    X_train = sub_train[feature_cols]
    y_train = sub_train[target].values.astype(float)
    X_val = sub_val[feature_cols]
    y_val = sub_val[target].values.astype(float)
    X_test = sub_test[feature_cols]
    y_test = sub_test[target].values.astype(float)

    q05, q95 = compute_tail_thresholds(y_train, 0.05, 0.95)

    # temperature（只用 train 计算）
    T = _compute_T_from_train(y_train)

    # negatives（只用 train labels）
    neg_idx = sample_negative_indices(len(y_train), CL_N_NEG, seed=SEED)
    neg_labels = y_train[neg_idx]

    obj = make_xgb_contrastive_obj(neg_labels=neg_labels, T=T, eps=CL_EPS)

    pre = build_preprocessor(numeric_cols, categorical_cols)
    pre, Xt, Xv, Xs = fit_transform_features(pre, X_train, X_val, X_test)

    # 拷贝 baseline 超参，只替换 objective（不改其它超参）
    params = dict(hparams.xgb_reg)
    params['objective'] = obj
    # eval_metric 用于 early stopping 的监控（不影响 reported metrics 的可比性）
    params.setdefault('eval_metric', 'rmse')

    model = train_xgb_regressor(Xt, y_train, Xv, y_val, params, hparams.early_stopping_rounds)
    y_pred = model.predict(Xs)

    diag = _compute_contrastive_diag_on_test(y_test=y_test, y_pred=y_pred, T=T, seed=SEED)
    return FitArtifacts(y_true=y_test, y_pred=y_pred, tail_q05=q05, tail_q95=q95, diag_test=diag, diag_name='p_i')


def run_contrastive_rt(df_in: pd.DataFrame, train_idx, val_idx, test_idx, target: str) -> FitArtifacts:
    # RT-CL：sklearn RandomForest 不支持自定义 objective，因此使用“对比式重加权”近似实现
    # w_i = 1 / (1 + Σ_j exp(-|y_i - y_j|/T))
    sub_train = df_in.iloc[train_idx].dropna(subset=[target])
    sub_val = df_in.iloc[val_idx].dropna(subset=[target])
    sub_test = df_in.iloc[test_idx].dropna(subset=[target])

    X_train = sub_train[feature_cols]
    y_train = sub_train[target].values.astype(float)
    X_val = sub_val[feature_cols]
    y_val = sub_val[target].values.astype(float)
    X_test = sub_test[feature_cols]
    y_test = sub_test[target].values.astype(float)

    q05, q95 = compute_tail_thresholds(y_train, 0.05, 0.95)

    T = _compute_T_from_train(y_train)
    w = compute_contrastive_weights_from_labels(y_train, n_neg=CL_N_NEG, T=T, seed=SEED)

    pre = build_preprocessor(numeric_cols, categorical_cols)
    pre, Xt, Xv, Xs = fit_transform_features(pre, X_train, X_val, X_test)

    model = train_rt_regressor(Xt, y_train, hparams.rf_reg, sample_weight=w)
    y_pred = model.predict(Xs)

    diag = _compute_contrastive_diag_on_test(y_test=y_test, y_pred=y_pred, T=T, seed=SEED)
    return FitArtifacts(y_true=y_test, y_pred=y_pred, tail_q05=q05, tail_q95=q95, diag_test=diag, diag_name='p_i')


print('Models ready (Baseline + Contrastive).')



Models ready (Baseline + Contrastive).


In [10]:
# 本 cell：定义“跑一次 split”的统一入口，保证同一 split 下 baseline 与 CL 实验组使用同一 train/val/test。


def pretty_print_split_sizes(df_in, train_idx, val_idx, test_idx, target: str):
    n_train = df_in.iloc[train_idx][target].notna().sum()
    n_val = df_in.iloc[val_idx][target].notna().sum()
    n_test = df_in.iloc[test_idx][target].notna().sum()
    print(f'[{target}] train/val/test (non-missing y): {n_train}/{n_val}/{n_test}')


def run_all_models_on_split(df_in: pd.DataFrame, train_idx, val_idx, test_idx, target: str):
    artifacts = {}

    # Baselines
    artifacts['XGB'] = run_baseline_xgb(df_in, train_idx, val_idx, test_idx, target)
    artifacts['RT']  = run_baseline_rt(df_in, train_idx, val_idx, test_idx, target)

    # Contrastive Learning experimental groups
    artifacts['XGB_CL'] = run_contrastive_xgb(df_in, train_idx, val_idx, test_idx, target)
    artifacts['RT_CL']  = run_contrastive_rt(df_in, train_idx, val_idx, test_idx, target)

    rows = []
    for name, art in artifacts.items():
        m = evaluate_with_tail(art.y_true, art.y_pred, art.tail_q05, art.tail_q95)
        row = {'Model': name, 'Target': target}
        row.update(m)
        rows.append(row)

    return pd.DataFrame(rows), artifacts


print('Runner ready.')



Runner ready.


In [11]:
# 本 cell：运行三种 split（Random / GroupSplit(grid_id) / CrossDomain(LOO:region_type or city_type)），
# 并对两目标分别对比 XGB / RT 及其 Contrastive Learning 实验组（XGB_CL / RT_CL）。

results_all = []
artifacts_all = {}  # (split_name, target) -> artifacts（用于画图）

MODEL_NAMES = ['XGB','RT','XGB_CL','RT_CL']

# ---- Split 1: Random split ----
train_idx_r, val_idx_r, test_idx_r = make_random_split(df, test_size=0.15, val_size=0.15, seed=SEED)
for target in target_cols:
    pretty_print_split_sizes(df, train_idx_r, val_idx_r, test_idx_r, target)
    res, arts = run_all_models_on_split(df, train_idx_r, val_idx_r, test_idx_r, target)
    res['Split'] = 'Random(70/15/15)'
    results_all.append(res)
    artifacts_all[('Random(70/15/15)', target)] = arts

# ---- Split 2: Group split by grid_id ----
if 'grid_id' in df.columns and df['grid_id'].nunique() >= 2:
    train_idx_g, val_idx_g, test_idx_g = make_group_split_by_grid(df, group_col='grid_id', test_size=0.15, val_size=0.15, seed=SEED)
    for target in target_cols:
        pretty_print_split_sizes(df, train_idx_g, val_idx_g, test_idx_g, target)
        res, arts = run_all_models_on_split(df, train_idx_g, val_idx_g, test_idx_g, target)
        res['Split'] = 'GroupSplit(grid_id)'
        results_all.append(res)
        artifacts_all[('GroupSplit(grid_id)', target)] = arts
else:
    print('⚠️ 未找到可用 grid_id，跳过 GroupSplit(grid_id)。')

# ---- Split 3: Cross-domain (Leave-One-Group-Out) ----
cross_group_col = choose_crossdomain_group_col(df)
print('Cross-domain group_col =', cross_group_col)

crossdomain_rows = []
crossdomain_pred_store = {}
for target in target_cols:
    for model_name in MODEL_NAMES:
        crossdomain_pred_store[(target, model_name)] = []

for train_val_idx, test_idx, held_out_group in iter_leave_one_group_out(df, cross_group_col):
    train_idx, val_idx = make_val_split_within_train(df, train_val_idx, val_size=0.15, seed=SEED)

    for target in target_cols:
        res, arts = run_all_models_on_split(df, train_idx, val_idx, test_idx, target)
        res['Split'] = f'CrossDomain(LOO:{cross_group_col})'
        res['HeldOutGroup'] = str(held_out_group)
        crossdomain_rows.append(res)

        for model_name, art in arts.items():
            crossdomain_pred_store[(target, model_name)].append(art)

crossdomain_df = pd.concat(crossdomain_rows, ignore_index=True)
results_all.append(crossdomain_df)

# ---- Cross-domain overall weighted summary ----
summary_rows = []
crossdomain_artifacts_for_plots = {}  # (target, model) -> aggregated FitArtifacts (for optional diagnostics)

for target in target_cols:
    for model_name in MODEL_NAMES:
        arts = crossdomain_pred_store[(target, model_name)]
        y_true_all = np.concatenate([a.y_true for a in arts])
        y_pred_all = np.concatenate([a.y_pred for a in arts])

        fold_metrics = []
        ns = []
        tail_ns = []
        for a in arts:
            fm = evaluate_with_tail(a.y_true, a.y_pred, a.tail_q05, a.tail_q95)
            fold_metrics.append(fm)
            ns.append(len(a.y_true))
            tail_ns.append(int(tail_mask(a.y_true, a.tail_q05, a.tail_q95).sum()))
        ns = np.array(ns, dtype=float)
        tail_ns = np.array(tail_ns, dtype=float)

        def wavg(key, weights):
            vals = np.array([fm[key] for fm in fold_metrics], dtype=float)
            return float(np.sum(vals * weights) / np.sum(weights)) if np.sum(weights) > 0 else np.nan

        row = {'Split':'CrossDomain(OverallWeighted)', 'HeldOutGroup':'ALL', 'Target':target, 'Model':model_name}
        row['All_MAE'] = wavg('All_MAE', ns)
        row['All_RMSE'] = wavg('All_RMSE', ns)
        row['All_AbsErr_P95'] = wavg('All_AbsErr_P95', ns)
        row['All_AbsErr_P99'] = wavg('All_AbsErr_P99', ns)
        row['Tail_Rate'] = float(np.sum(tail_ns) / np.sum(ns))

        if np.sum(tail_ns) > 0:
            row['Tail_MAE'] = wavg('Tail_MAE', tail_ns)
            row['Tail_RMSE'] = wavg('Tail_RMSE', tail_ns)
            row['Tail_AbsErr_P95'] = wavg('Tail_AbsErr_P95', tail_ns)
            row['Tail_AbsErr_P99'] = wavg('Tail_AbsErr_P99', tail_ns)
        else:
            row['Tail_MAE'] = np.nan
            row['Tail_RMSE'] = np.nan
            row['Tail_AbsErr_P95'] = np.nan
            row['Tail_AbsErr_P99'] = np.nan

        row['All_R2'] = float(r2_score(y_true_all, y_pred_all))
        y_true_tail_all = np.concatenate([
            a.y_true[tail_mask(a.y_true, a.tail_q05, a.tail_q95)]
            for a in arts if tail_mask(a.y_true, a.tail_q05, a.tail_q95).sum() > 0
        ])
        y_pred_tail_all = np.concatenate([
            a.y_pred[tail_mask(a.y_true, a.tail_q05, a.tail_q95)]
            for a in arts if tail_mask(a.y_true, a.tail_q05, a.tail_q95).sum() > 0
        ])
        row['Tail_R2'] = float(r2_score(y_true_tail_all, y_pred_tail_all)) if len(y_true_tail_all) > 1 else np.nan

        summary_rows.append(row)

        # 仅对 CL 模型保存诊断量（将各 fold 的 diag_test 拼接）
        if model_name in ['XGB_CL', 'RT_CL']:
            diag_all = []
            for a in arts:
                if a.diag_test is not None:
                    diag_all.append(a.diag_test)
            diag_all = np.concatenate(diag_all) if len(diag_all) > 0 else None
            # 为了画 tail vs non-tail（诊断层面），这里用 overall 的 5/95 分位（仅用于图，不参与训练/指标）
            q05_all, q95_all = compute_tail_thresholds(y_true_all, 0.05, 0.95)
            crossdomain_artifacts_for_plots[(target, model_name)] = FitArtifacts(
                y_true=y_true_all,
                y_pred=y_pred_all,
                tail_q05=q05_all,
                tail_q95=q95_all,
                diag_test=diag_all,
                diag_name='p_i'
            )

summary_df = pd.DataFrame(summary_rows)

results_df = pd.concat(results_all + [summary_df], ignore_index=True)
print('Done. Total result rows:', len(results_df))
display(results_df.head(10))



[dVIIRS] train/val/test (non-missing y): 17832/3820/3844
[dWorldPop] train/val/test (non-missing y): 17832/3820/3844
[dVIIRS] train/val/test (non-missing y): 17846/3825/3825
[dWorldPop] train/val/test (non-missing y): 17846/3825/3825
Cross-domain group_col = region_type
Done. Total result rows: 64


,Model,Target,All_MAE,All_RMSE,All_R2,All_AbsErr_P95,All_AbsErr_P99,Tail_Rate,Tail_MAE,Tail_RMSE,Tail_R2,Tail_AbsErr_P95,Tail_AbsErr_P99,Split,HeldOutGroup
0,XGB,dVIIRS,2.482697,4.388171,0.025539,8.380994,18.008632,0.105359,9.224798,11.413658,0.141433,21.700452,34.863604,Random(70/15/15),NaN
1,RT,dVIIRS,2.391630,4.299768,0.064406,8.129128,17.066146,0.105359,9.208009,11.411292,0.141789,22.018026,37.943647,Random(70/15/15),NaN
2,XGB_CL,dVIIRS,3.043759,5.299567,-0.421275,10.533958,20.724603,0.105359,9.351055,12.548490,-0.037785,26.315299,41.098062,Random(70/15/15),NaN
3,RT_CL,dVIIRS,2.415205,4.345533,0.044384,8.013601,16.684942,0.105359,9.429829,11.634148,0.107941,23.047583,37.467544,Random(70/15/15),NaN
4,XGB,dWorldPop,8.795423,21.976476,0.835819,27.117124,92.214006,0.105099,34.191392,61.047571,0.831624,128.687043,223.382698,Random(70/15/15),NaN
5,RT,dWorldPop,9.219622,25.247793,0.783303,36.580761,107.457868,0.105099,42.230358,71.571008,0.768571,146.052299,294.915122,Random(70/15/15),NaN
6,XGB_CL,dWorldPop,33.696137,58.240923,-0.153089,106.537441,185.497316,0.105099,90.199958,136.270061,0.161034,262.073312,427.939237,Random(70/15/15),NaN
7,RT_CL,dWorldPop,8.424460,22.809179,0.823142,31.269352,97.315238,0.105099,37.476998,64.606284,0.811421,143.778560,221.045554,Random(70/15/15),NaN
8,XGB,dVIIRS,2.458283,4.491183,0.003108,8.447606,18.265755,0.104837,9.646604,12.090744,0.074556,22.074576,34.259021,GroupSplit(grid_id),NaN
9,RT,dVIIRS,2.379361,4.346324,0.066379,8.282585,17.188030,0.104837,9.501227,11.768906,0.123169,21.437032,34.644029,GroupSplit(grid_id),NaN


In [12]:
# 本 cell：把结果表直接整理成 RESULT1（长表）和 RESULT2（pivot 总表），并尽量避免输出被省略

import pandas as pd
import numpy as np

# 1) 尽量取消 pandas 输出截断（Jupyter 会更完整显示）
pd.set_option('display.max_rows', 5000)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 2000)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.expand_frame_repr', False)

# 2) 你论文需要的关键列
key_cols = [
    'Split','HeldOutGroup','Target','Model',
    'All_MAE','All_RMSE','All_R2','All_AbsErr_P95','All_AbsErr_P99',
    'Tail_MAE','Tail_RMSE','Tail_R2','Tail_AbsErr_P95','Tail_AbsErr_P99',
    'Tail_Rate'
]

# 若缺列则补 NaN，保证稳健
for c in key_cols:
    if c not in results_df.columns:
        results_df[c] = np.nan

# 3) RESULT1：完整长表（含跨域每个 HeldOutGroup 的明细）
RESULT1 = results_df[key_cols].copy()

# split 排序（如果 cross_group_col 没定义，做个兜底）
try:
    cd_name = f'CrossDomain(LOO:{cross_group_col})'
except Exception:
    cd_name = 'CrossDomain(LOO:UNKNOWN)'

split_order = ['Random(70/15/15)', 'GroupSplit(grid_id)', cd_name, 'CrossDomain(OverallWeighted)']
RESULT1['Split'] = pd.Categorical(RESULT1['Split'], categories=split_order, ordered=True)

RESULT1 = RESULT1.sort_values(['Split','Target','Model','HeldOutGroup']).reset_index(drop=True)

# 4) RESULT2：pivot 总表（只汇总整体行：HeldOutGroup 为空或 ALL）
pivot_src = RESULT1[(RESULT1['HeldOutGroup'].isna()) | (RESULT1['HeldOutGroup'].isin(['ALL']))].copy()
pivot_src = pivot_src[pivot_src['Split'].isin(['Random(70/15/15)', 'GroupSplit(grid_id)', 'CrossDomain(OverallWeighted)'])].copy()

RESULT2 = pivot_src.pivot_table(
    index=['Split','Target'],
    columns='Model',
    values=[
        'All_MAE','All_RMSE','All_R2',
        'Tail_MAE','Tail_RMSE','Tail_R2',
        'All_AbsErr_P99','Tail_AbsErr_P99'
    ],
    aggfunc='first'
)

# 5) 直接输出两张表（如果仍觉得太长，你可以 RESULT1.to_csv / Excel 导出）
print("RESULT1 shape:", RESULT1.shape)
display(RESULT1)

print("RESULT2 shape:", RESULT2.shape)
display(RESULT2)

# 本 cell：把 RESULT1 / RESULT2 保存为当前路径下的 Excel（.xlsx）

import os

out_path = "RESULTS_CL.xlsx"  # 当前路径
print("将保存到：", os.path.abspath(out_path))

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    RESULT1.to_excel(writer, sheet_name="RESULT1_long", index=False)
    # RESULT2 是 pivot，多级列索引，先 reset_index 方便阅读
    RESULT2_reset = RESULT2.copy()
    RESULT2_reset.to_excel(writer, sheet_name="RESULT2_pivot")

print("✅ 已保存：", out_path)


RESULT1 shape: (64, 15)


,Split,HeldOutGroup,Target,Model,All_MAE,All_RMSE,All_R2,All_AbsErr_P95,All_AbsErr_P99,Tail_MAE,Tail_RMSE,Tail_R2,Tail_AbsErr_P95,Tail_AbsErr_P99,Tail_Rate
0,Random(70/15/15),NaN,dVIIRS,RT,2.391630,4.299768,0.064406,8.129128,17.066146,9.208009,11.411292,0.141789,22.018026,37.943647,0.105359
1,Random(70/15/15),NaN,dVIIRS,RT_CL,2.415205,4.345533,0.044384,8.013601,16.684942,9.429829,11.634148,0.107941,23.047583,37.467544,0.105359
2,Random(70/15/15),NaN,dVIIRS,XGB,2.482697,4.388171,0.025539,8.380994,18.008632,9.224798,11.413658,0.141433,21.700452,34.863604,0.105359
3,Random(70/15/15),NaN,dVIIRS,XGB_CL,3.043759,5.299567,-0.421275,10.533958,20.724603,9.351055,12.548490,-0.037785,26.315299,41.098062,0.105359
4,Random(70/15/15),NaN,dWorldPop,RT,9.219622,25.247793,0.783303,36.580761,107.457868,42.230358,71.571008,0.768571,146.052299,294.915122,0.105099
5,Random(70/15/15),NaN,dWorldPop,RT_CL,8.424460,22.809179,0.823142,31.269352,97.315238,37.476998,64.606284,0.811421,143.778560,221.045554,0.105099
6,Random(70/15/15),NaN,dWorldPop,XGB,8.795423,21.976476,0.835819,27.117124,92.214006,34.191392,61.047571,0.831624,128.687043,223.382698,0.105099
7,Random(70/15/15),NaN,dWorldPop,XGB_CL,33.696137,58.240923,-0.153089,106.537441,185.497316,90.199958,136.270061,0.161034,262.073312,427.939237,0.105099
8,GroupSplit(grid_id),NaN,dVIIRS,RT,2.379361,4.346324,0.066379,8.282585,17.188030,9.501227,11.768906,0.123169,21.437032,34.644029,0.104837
9,GroupSplit(grid_id),NaN,dVIIRS,RT_CL,2.364442,4.313575,0.080395,8.272934,17.186708,9.619455,11.754836,0.125264,22.119967,35.230106,0.104837


RESULT2 shape: (6, 32)


All_AbsErr_P99                                        All_MAE                                     All_R2                                 All_RMSE                                  Tail_AbsErr_P99                                       Tail_MAE                                    Tail_R2                                 Tail_RMSE                                    
Model                                              RT       RT_CL         XGB      XGB_CL         RT      RT_CL        XGB     XGB_CL        RT     RT_CL       XGB    XGB_CL         RT      RT_CL        XGB     XGB_CL              RT       RT_CL         XGB      XGB_CL         RT      RT_CL        XGB     XGB_CL        RT     RT_CL       XGB    XGB_CL          RT       RT_CL         XGB      XGB_CL
Split                        Target                                                                                                                                                                                                                                                                                                                                                                              
Random(70/15/15)             dVIIRS         17.066146   16.684942   18.008632   20.724603   2.391630   2.415205   2.482697   3.043759  0.064406  0.044384  0.025539 -0.421275   4.299768   4.345533   4.388171   5.299567       37.943647   37.467544   34.863604   41.098062   9.208009   9.429829   9.224798   9.351055  0.141789  0.107941  0.141433 -0.037785   11.411292   11.634148   11.413658   12.548490
                             dWorldPop     107.457868   97.315238   92.214006  185.497316   9.219622   8.424460   8.795423  33.696137  0.783303  0.823142  0.835819 -0.153089  25.247793  22.809179  21.976476  58.240923      294.915122  221.045554  223.382698  427.939237  42.230358  37.476998  34.191392  90.199958  0.768571  0.811421  0.831624  0.161034   71.571008   64.606284   61.047571  136.270061
GroupSplit(grid_id)          dVIIRS         17.188030   17.186708   18.265755   21.429542   2.379361   2.364442   2.458283   2.999803  0.066379  0.080395  0.003108 -0.428695   4.346324   4.313575   4.491183   5.376584       34.644029   35.230106   34.259021   43.028362   9.501227   9.619455   9.646604   9.916993  0.123169  0.125264  0.074556 -0.069595   11.768906   11.754836   12.090744   12.998333
                             dWorldPop     186.698665  161.399573  183.199248  183.828805  19.867876  19.966776  20.391979  33.846650  0.470775  0.523127  0.522954 -0.040950  44.401910  42.148572  42.156209  62.272517      418.460841  453.391530  485.809050  733.996103  85.420577  86.249990  84.552737  94.115544  0.482772  0.531909  0.511091  0.119376  120.243220  114.389100  116.905131  156.896981
CrossDomain(OverallWeighted) dVIIRS         17.016296   16.848157   18.139283   19.791370   2.392946   2.392735   2.547752   3.062753 -0.035674 -0.028138 -0.180264 -0.528749   4.321890   4.308076   4.607163   5.234195       38.104071   38.255909   41.054169   40.977902   9.243679   9.361426   9.428214   9.535462  0.023146  0.020333 -0.062124 -0.156701   11.643871   11.664071   12.142113   12.673893
                             dWorldPop     182.168715  177.993369  171.875200  187.053196  20.188207  20.138013  20.503442  34.487859  0.333271  0.335306  0.351946 -0.196131  42.199849  42.288352  41.867425  57.145840      414.375039  417.775235  427.686363  515.613478  80.126927  80.404889  77.769350  90.202566  0.368555  0.360934  0.369062  0.127855  110.650115  111.070066  110.223248  129.935514

将保存到： /Users/huangwenqin/Desktop/weird code/RESULTS_CL.xlsx
✅ 已保存： RESULTS_CL.xlsx


In [13]:
import matplotlib.pyplot as plt
import numpy as np

def plot_pred_vs_true(y_true, y_pred, q05=None, q95=None, title='Pred vs True', highlight_tail=True):
    fig, ax = plt.subplots()

    if highlight_tail and (q05 is not None) and (q95 is not None):
        m = tail_mask(y_true, q05, q95)
        ax.scatter(y_true[~m], y_pred[~m], s=8, alpha=0.4, marker='o', label='Non-tail')
        ax.scatter(y_true[m],  y_pred[m],  s=14, alpha=0.8, marker='x', label='Tail')
    else:
        ax.scatter(y_true, y_pred, s=8, alpha=0.5)

    # y = x reference line
    mn = float(np.min([np.min(y_true), np.min(y_pred)]))
    mx = float(np.max([np.max(y_true), np.max(y_pred)]))
    ax.plot([mn, mx], [mn, mx], linestyle='--', linewidth=1, alpha=0.8, label='y=x')

    ax.set_xlabel('True')
    ax.set_ylabel('Pred')
    ax.set_title(title)
    ax.grid(True, alpha=0.2)
    ax.legend()
    fig.tight_layout()
    return fig



def plot_residual_hist(y_true, y_pred, q05=None, q95=None, title='Residual distribution'):
    res = (y_pred - y_true)

    fig1, ax1 = plt.subplots()
    ax1.hist(res, bins=60, alpha=0.75)
    ax1.set_title(title)
    ax1.set_xlabel('Residual (pred - true)')
    ax1.set_ylabel('Count')
    ax1.grid(True, alpha=0.2)
    fig1.tight_layout()

    figs = [fig1]

    if (q05 is not None) and (q95 is not None):
        m = tail_mask(y_true, q05, q95)
        if m.sum() > 0:
            fig2, ax2 = plt.subplots()
            ax2.hist(res[m], bins=60, alpha=0.75)
            ax2.set_title(title + ' (Tail only)')
            ax2.set_xlabel('Residual (pred - true)')
            ax2.set_ylabel('Count')
            ax2.grid(True, alpha=0.2)
            fig2.tight_layout()
            figs.append(fig2)

    return figs



def plot_contrastive_diagnostics(art: FitArtifacts, title_prefix='Contrastive diagnostics'):
    # art.diag_test 默认存 p_i（对比式概率）
    if (art.diag_test is None) or (len(art.diag_test) == 0):
        return []

    v = np.asarray(art.diag_test, dtype=float)
    figs = []

    fig1, ax1 = plt.subplots()
    ax1.hist(v, bins=50, alpha=0.75)
    ax1.set_title(title_prefix + f' - {art.diag_name or "diag"} distribution')
    ax1.set_xlabel(art.diag_name or 'diag')
    ax1.set_ylabel('Count')
    ax1.grid(True, alpha=0.2)
    fig1.tight_layout()
    figs.append(fig1)

    # tail vs non-tail mean（如果提供了 tail 阈值）
    if not (np.isnan(art.tail_q05) or np.isnan(art.tail_q95)):
        m = tail_mask(art.y_true, art.tail_q05, art.tail_q95)
        mean_tail = float(np.mean(v[m])) if m.sum() > 0 else np.nan
        mean_nontail = float(np.mean(v[~m])) if (~m).sum() > 0 else np.nan

        fig2, ax2 = plt.subplots()
        ax2.bar(['Non-tail mean', 'Tail mean'], [mean_nontail, mean_tail])
        ax2.set_title(title_prefix + f' - mean {art.diag_name or "diag"} on tail vs non-tail')
        ax2.set_ylabel(f'Mean {art.diag_name or "diag"}')
        ax2.grid(True, axis='y', alpha=0.2)
        fig2.tight_layout()
        figs.append(fig2)

    return figs



In [15]:
import os, re

OUTDIR = "paper_figs_cl"
os.makedirs(OUTDIR, exist_ok=True)

def safe(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", s)

def save_fig(fig, name, dpi=300):
    path = os.path.join(OUTDIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return path


In [16]:
# 本 cell：为每个 split + target 生成关键图。
# 为什么：论文中常用 Random 与 GroupSplit 做代表性展示；cross-domain 可选展示 CL 诊断量分布。


def run_plots_for_setting(split_name: str, target: str):
    arts = artifacts_all[(split_name, target)]

    for model_key in ['XGB', 'XGB_CL', 'RT', 'RT_CL']:
        art = arts[model_key]

        fig = plot_pred_vs_true(art.y_true, art.y_pred, art.tail_q05, art.tail_q95,
                                title=f'[{split_name}] {model_key} - {target}: Pred vs True')
        save_fig(fig, f"{safe(split_name)}__{safe(target)}__{safe(model_key)}__pred_true.png")

        for i, fig in enumerate(plot_residual_hist(art.y_true, art.y_pred, art.tail_q05, art.tail_q95,
                                                   title=f'[{split_name}] {model_key} - {target}: Residuals')):
            save_fig(fig, f"{safe(split_name)}__{safe(target)}__{safe(model_key)}__resid_{i}.png")

        # 仅对 CL 模型输出诊断图
        if art.diag_test is not None:
            for i, fig in enumerate(plot_contrastive_diagnostics(art, title_prefix=f'[{split_name}] {model_key} - {target}')):
                save_fig(fig, f"{safe(split_name)}__{safe(target)}__{safe(model_key)}__diag_{i}.png")


for target in target_cols:
    run_plots_for_setting('Random(70/15/15)', target)

if ('GroupSplit(grid_id)', target_cols[0]) in artifacts_all:
    for target in target_cols:
        run_plots_for_setting('GroupSplit(grid_id)', target)

# Cross-domain overall diagnostic（可选）
for target in target_cols:
    for model_key in ['XGB_CL', 'RT_CL']:
        k = (target, model_key)
        if k in crossdomain_artifacts_for_plots:
            art = crossdomain_artifacts_for_plots[k]
            for i, fig in enumerate(plot_contrastive_diagnostics(art, title_prefix=f'[CrossDomain Overall] {model_key} - {target}')):
                save_fig(fig, f"CrossDomainOverall__{safe(target)}__{safe(model_key)}__diag_{i}.png")



In [ ]:
# 本 cell：导出结果表到 CSV，方便论文直接引用与复现实验记录。
out_csv = '/mnt/data/transluxpop_contrastive_results_table.csv'
# 建议导出 RESULT1（长表，含 cross-domain 的 HeldOutGroup 明细）。
RESULT1.to_csv(out_csv, index=False)
print('Saved:', out_csv)


## 备注与可扩展点（建议写进论文/附录）

- **温度 T 的选择**：本 Notebook 使用 `T = std(y_train) * CL_TEMP_FACTOR` 的自适应方案，避免两个目标（dVIIRS / dWorldPop）尺度差异导致对比项失效。
- **负样本数量 K（CL_N_NEG）**：K 越大越接近 full-batch contrastive，但训练更慢；建议在论文里报告你最终采用的 K，并在附录做一个小的敏感性分析（例如 K=8/16/32）。
- **对比式诊断量 p_i**：Notebook 会在 test 上（内部采样 negatives）计算 p_i 并画分布图；
  - 如果 tail 的平均 p_i 明显低于 non-tail，说明 tail 样本在“与其它 y 区分”上更难，可能需要更大 T 或更多 negatives。
  - 如果 p_i 普遍接近 1，说明负样本与 anchor 的距离普遍很大（对比项过弱）；可考虑降低 T 或增大 negatives。
- **RT-CL 的实现说明**：sklearn RandomForest 不支持自定义 objective，因此用对比式目标推导的样本权重（contrastive reweighting）近似实现“削弱 head、强调 tail”。
  若你希望更严格地实现对比式目标，可在附录补充一个可微模型（如小 MLP）作为补充对照。

